In [ ]:
import pandas as pd

df = pd.read_csv("food_expiry_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (177555, 7)


,Food_type,Storage_condition,Temperature,Humidity,Packaging,Date_of_purchase,Expiry_days
0,Eggs,Fridge,-3.9,59,Sealed,01-01-2023,19
1,Meat,Fridge,-12.6,40,Sealed,01-01-2023,30
2,Fruit,Freezer,30.1,58,Sealed,01-01-2023,10
3,Eggs,Room,6.4,70,Open,01-01-2023,10
4,Vegetable,Room,8.3,72,Sealed,01-01-2023,5


In [ ]:
print(df.isnull().sum())

Food_type            1
Storage_condition    4
Temperature          0
Humidity             0
Packaging            4
Date_of_purchase     0
Expiry_days          0
dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177555 entries, 0 to 177554
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Food_type          177554 non-null  object 
 1   Storage_condition  177551 non-null  object 
 2   Temperature        177555 non-null  float64
 3   Humidity           177555 non-null  int64  
 4   Packaging          177551 non-null  object 
 5   Date_of_purchase   177555 non-null  object 
 6   Expiry_days        177555 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 9.5+ MB


In [ ]:
print("Minimum:", df["Expiry_days"].min())
print("Maximum:", df["Expiry_days"].max())

print("\nExpiry days summary:")
print(df["Expiry_days"].describe())

Minimum: 1
Maximum: 187

Expiry days summary:
count    177555.000000
mean         16.584658
std          17.841610
min           1.000000
25%           5.000000
50%          10.000000
75%          21.000000
max         187.000000
Name: Expiry_days, dtype: float64


#Data Preprocessing.

#Remove missing values

In [ ]:
df = df.dropna()

print("Dataset shape:", df.shape)

Dataset shape: (177546, 7)


#Convert expiry days into categories

In [ ]:
def categorize_expiry(days):
    if days <= 7:
        return "Short"
    elif days <= 15:
        return "Medium"
    else:
        return "Long"

df["Expiry_category"] = df["Expiry_days"].apply(categorize_expiry)

print(df[["Expiry_days", "Expiry_category"]].head())

   Expiry_days Expiry_category
0           19            Long
1           30            Long
2           10          Medium
3           10          Medium
4            5           Short


#Check the category distribution

In [ ]:
print(df["Expiry_category"].value_counts())

Expiry_category
Short     67631
Long      62927
Medium    46988
Name: count, dtype: int64


#Select the features

In [ ]:
data = df[[
    "Food_type",
    "Storage_condition",
    "Packaging",
    "Temperature",
    "Humidity",
    "Expiry_category"
]].copy()

print(data.head())

   Food_type Storage_condition Packaging  Temperature  Humidity  \
0       Eggs            Fridge    Sealed         -3.9        59   
1       Meat            Fridge    Sealed        -12.6        40   
2      Fruit           Freezer    Sealed         30.1        58   
3       Eggs              Room      Open          6.4        70   
4  Vegetable              Room    Sealed          8.3        72   

  Expiry_category  
0            Long  
1            Long  
2          Medium  
3          Medium  
4           Short  


#Prepare data for the Bayesian Network

In [ ]:
data["Temperature"] = pd.cut(
    data["Temperature"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

data["Humidity"] = pd.cut(
    data["Humidity"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

print(data.head())

   Food_type Storage_condition Packaging Temperature Humidity Expiry_category
0       Eggs            Fridge    Sealed         Low   Medium            Long
1       Meat            Fridge    Sealed         Low      Low            Long
2      Fruit           Freezer    Sealed        High   Medium          Medium
3       Eggs              Room      Open      Medium     High          Medium
4  Vegetable              Room    Sealed      Medium     High           Short


In [ ]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 13.7 MB/s eta 0:00:00


#Build the Bayesian Network

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork

model = DiscreteBayesianNetwork([
    ("Food_type", "Expiry_category"),
    ("Storage_condition", "Expiry_category"),
    ("Packaging", "Expiry_category"),
    ("Temperature", "Expiry_category"),
    ("Humidity", "Expiry_category")
])

model.fit(data)

print("Bayesian Network trained successfully!")

Bayesian Network trained successfully!


#Check the model

In [ ]:
print("Model nodes:", model.nodes())
print("Model edges:", model.edges())

Model nodes: ['Food_type', 'Expiry_category', 'Storage_condition', 'Packaging', 'Temperature', 'Humidity']
Model edges: [('Food_type', 'Expiry_category'), ('Storage_condition', 'Expiry_category'), ('Packaging', 'Expiry_category'), ('Temperature', 'Expiry_category'), ('Humidity', 'Expiry_category')]


#Step 4: Making predictions using the Bayesian Network.

#Import the inference library

In [ ]:
from pgmpy.inference import VariableElimination

inference = VariableElimination(model)

#Predict the expiry category

In [ ]:
sample = data.iloc[0]

evidence = {
    "Food_type": sample["Food_type"],
    "Storage_condition": sample["Storage_condition"],
    "Packaging": sample["Packaging"],
    "Temperature": sample["Temperature"],
    "Humidity": sample["Humidity"]
}

result = inference.query(
    variables=["Expiry_category"],
    evidence=evidence
)

print(result)

+-------------------------+------------------------+
| Expiry_category         |   phi(Expiry_category) |
+=========================+========================+
| Expiry_category(Long)   |                 0.8275 |
+-------------------------+------------------------+
| Expiry_category(Medium) |                 0.1725 |
+-------------------------+------------------------+
| Expiry_category(Short)  |                 0.0000 |
+-------------------------+------------------------+


#Display the predicted category

In [ ]:
predicted_category = result.state_names["Expiry_category"][
    result.values.argmax()
]

print("Predicted expiry category:", predicted_category)
print("Actual expiry category:", sample["Expiry_category"])

Predicted expiry category: Long
Actual expiry category: Long


#Step 5: Evaluate the model.

#Split the dataset

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    data, test_size=0.2, random_state=42
)

print("Training data:", len(train_data))
print("Testing data:", len(test_data))

Training data: 142036
Testing data: 35510


#Train the model on training data

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork

model = DiscreteBayesianNetwork([
    ("Food_type", "Expiry_category"),
    ("Storage_condition", "Expiry_category"),
    ("Packaging", "Expiry_category"),
    ("Temperature", "Expiry_category"),
    ("Humidity", "Expiry_category")
])

model.fit(train_data)

print("Model trained successfully!")

Model trained successfully!


#Evaluate the model

In [ ]:
from pgmpy.inference import VariableElimination
from sklearn.metrics import accuracy_score, classification_report

inference = VariableElimination(model)

y_true = []
y_pred = []

for _, row in test_data.iterrows():
    evidence = {
        "Food_type": row["Food_type"],
        "Storage_condition": row["Storage_condition"],
        "Packaging": row["Packaging"],
        "Temperature": row["Temperature"],
        "Humidity": row["Humidity"]
    }

    result = inference.query(
        variables=["Expiry_category"],
        evidence=evidence
    )

    prediction = result.state_names["Expiry_category"][
        result.values.argmax()
    ]

    y_true.append(row["Expiry_category"])
    y_pred.append(prediction)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, zero_division=0))

Accuracy: 0.7012954097437342
              precision    recall  f1-score   support

        Long       0.72      0.87      0.79     12622
      Medium       0.49      0.42      0.45      9313
       Short       0.81      0.74      0.77     13575

    accuracy                           0.70     35510
   macro avg       0.68      0.68      0.67     35510
weighted avg       0.70      0.70      0.69     35510

